# 06 Awareness & Literacy
**Owner:** Yasemin Gunindi

This notebook implements the Awareness & Literacy layer for the AI early-warning system. It does not change the model prediction. Instead, it explains how students and educators should interpret prediction outputs responsibly.

The notebook first presents general model information, including the system’s intended use, data sources, limitations, and responsible use conditions. It then demonstrates two role-based views: a student view and an educator view. The student view focuses on individual understanding and rights, while the educator view supports responsible interpretation, human review, and supportive intervention planning.


In [25]:
from pathlib import Path
import sys
import joblib
import pandas as pd
import numpy as np
import importlib

# Find project root robustly
current = Path.cwd().resolve()

for path in [current, *current.parents]:
    if (path / "src").exists() and (path / "data").exists():
        PROJECT_ROOT = path
        break
else:
    raise FileNotFoundError("Project root not found. Make sure you are inside the project folder.")

sys.path.insert(0, str(PROJECT_ROOT))

import src.awareness as awareness
importlib.reload(awareness)

from src.awareness import (
    print_general_model_notice,
    get_role_guidance,
    annotate_prediction,
    generate_educator_briefing,
    generate_comprehension_check
)

data_dir = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data directory:", data_dir)

Project root: C:\Users\suuser\PycharmProjects\CS540-ethics-in-data-ai-project
Data directory: C:\Users\suuser\PycharmProjects\CS540-ethics-in-data-ai-project\data


In [26]:
model = joblib.load(data_dir / "base_model.pkl")
X_test = joblib.load(data_dir / "X_test.pkl")
y_test = joblib.load(data_dir / "y_test.pkl")

print("Model loaded:", type(model))
print("X_test shape:", X_test.shape)
print("y_test length:", len(y_test))

Model loaded: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
X_test shape: (6519, 11)
y_test length: 6519


In [27]:
y_pred = model.predict(X_test)

if hasattr(model, "predict_proba"):
    y_proba = model.predict_proba(X_test)
    model_confidence = y_proba.max(axis=1)
else:
    model_confidence = np.repeat(np.nan, len(y_pred))

prediction_output = pd.DataFrame({
    "student_id": X_test.index,
    "model_prediction": y_pred,
    "model_confidence": model_confidence
})

prediction_output.head()

,student_id,model_prediction,model_confidence
0,23252,0,0.888164
1,30849,1,1.000000
2,21169,1,0.577619
3,30521,0,0.891392
4,29539,0,0.859759


In [28]:
# 0 = Pass, 1 = At-Risk based on the base model classification report
def map_to_risk_label(prediction):
    if prediction == 0:
        return "Pass"
    elif prediction == 1:
        return "At-Risk"
    else:
        return str(prediction)

prediction_output["risk_label"] = prediction_output["model_prediction"].apply(map_to_risk_label)

risk_class = 1  # 1 = At-Risk
risk_class_index = list(model.classes_).index(risk_class)
prediction_output["risk_probability"] = y_proba[:, risk_class_index]

prediction_output.head()

,student_id,model_prediction,model_confidence,risk_label,risk_probability
0,23252,0,0.888164,Pass,0.111836
1,30849,1,1.000000,At-Risk,1.000000
2,21169,1,0.577619,At-Risk,0.577619
3,30521,0,0.891392,Pass,0.108608
4,29539,0,0.859759,Pass,0.140241


In [29]:
print_general_model_notice()

AI EARLY-WARNING SYSTEM: GENERAL MODEL INFORMATION

This system uses a machine learning model to estimate whether a student may need early academic support.

The prediction is not a final verdict about the student.
It should be understood as a decision-support signal.

The model was trained on historical educational data. It may use information such as:
- VLE click activity
- Assessment scores
- Registration records
- Course information

Important limitations:
- The model cannot fully understand personal circumstances.
- The model cannot know recent life events, health issues, motivation, family context, or financial stress.
- The model may reflect patterns and inequalities present in historical educational data.
- A numerical score can look objective, but it is still produced by a model with limitations.

Responsible use:
- Use predictions to offer support, not punishment.
- Do not use predictions as the only basis for decisions.
- A human advisor or educator should review the predict

Demo 1: Student View

In [30]:
# Student role guidance
student_role = "student"

print(get_role_guidance(student_role))

STUDENT INFORMATION

What this system does:
- The system estimates whether you may need academic support.
- If you are marked as "At-Risk", this does not mean you will fail.
- It means the system thinks additional support may be useful.

What you should know:
- The prediction is based on data patterns.
- It may not include your personal situation.
- It may not understand why your activity or scores changed.
- The prediction should not define your ability or potential.

Your rights:
- You can ask for a human review.
- You can ask what information contributed to the prediction.
- You can explain your situation to an advisor if you want.
- You do not have to disclose personal circumstances.
- The prediction should not be used to punish or exclude you.

Best interpretation:
- Treat the prediction as an invitation to support, not as a judgment.


In [31]:
# Student-specific prediction example
student_sample = prediction_output.iloc[0]

print(
    annotate_prediction(
        student_id=student_sample["student_id"],
        risk_label=student_sample["risk_label"],
        confidence=student_sample["model_confidence"],
        role=student_role
    )
)

Student ID: 23252
Predicted outcome: Pass
Model confidence: 88.8%

Interpretation:
This output is a decision-support signal, not a final verdict.

For the student:
- If the result is 'At-Risk', it means support may be helpful.
- It does not mean you will fail.
- You may request human review.
- You may ask what information contributed to the prediction.
- The prediction should not be used to punish or exclude you.


In [32]:
generate_comprehension_check(student_role)

,question,correct_answer
0,Is the model prediction a final decision?,No. It is a decision-support signal that requi...
1,Can the model fully understand personal circum...,"No. The model may miss health, family, financi..."
2,Should an 'At-Risk' prediction be used for pun...,"No. It should be used to offer support, not pu..."
3,What can a student do if they disagree with th...,They can request human review and ask what inf...


Demo 2: Educator View

In [33]:
# Educator role guidance
educator_role = "educator"

print(get_role_guidance(educator_role))

EDUCATOR INFORMATION

What this system does:
- The system estimates which students may benefit from early academic support.
- It can help prioritize outreach, advising, or additional resources.

What educators should avoid:
- Do not treat the prediction as a final truth.
- Do not use the score as the only basis for intervention.
- Do not stigmatize students marked as "At-Risk".
- Do not assume that low engagement always means low effort.

Responsible interpretation:
- Use the prediction as a prompt for supportive outreach.
- Combine the model output with qualitative knowledge of the student.
- Consider whether the prediction may be affected by data quality or missing context.
- Give students a chance to clarify or contest the prediction.
- Monitor whether some groups are more often misclassified or over-flagged.

Best interpretation:
- The model can help identify possible support needs, but educators remain responsible for contextual judgment.


In [34]:
educator_summary = pd.DataFrame({
    "metric": [
        "Total number of students",
        "Predicted Pass",
        "Predicted At-Risk",
        "Mean risk probability",
        "Maximum risk probability"
    ],
    "value": [
        len(prediction_output),
        (prediction_output["risk_label"] == "Pass").sum(),
        (prediction_output["risk_label"] == "At-Risk").sum(),
        round(prediction_output["risk_probability"].mean(), 3),
        round(prediction_output["risk_probability"].max(), 3)
    ]
})

educator_summary

,metric,value
0,Total number of students,6519.00
1,Predicted Pass,3340.00
2,Predicted At-Risk,3179.00
3,Mean risk probability,0.52
4,Maximum risk probability,1.00


In [35]:
highest_risk_for_supportive_review = (
    prediction_output
    .sort_values("risk_probability", ascending=False)
    .head(10)
)

highest_risk_for_supportive_review[
    ["student_id", "risk_label", "model_confidence", "risk_probability"]
]

,student_id,risk_label,model_confidence,risk_probability
1,30849,At-Risk,1.0,1.0
7,11315,At-Risk,1.0,1.0
6491,25044,At-Risk,1.0,1.0
6489,8394,At-Risk,1.0,1.0
13,16201,At-Risk,1.0,1.0
26,5669,At-Risk,1.0,1.0
24,21658,At-Risk,1.0,1.0
1736,19532,At-Risk,1.0,1.0
1803,21946,At-Risk,1.0,1.0
1805,13135,At-Risk,1.0,1.0


In [36]:
educator_sample = highest_risk_for_supportive_review.iloc[0]

print(
    annotate_prediction(
        student_id=educator_sample["student_id"],
        risk_label=educator_sample["risk_label"],
        confidence=educator_sample["model_confidence"],
        role=educator_role
    )
)

Student ID: 30849
Predicted outcome: At-Risk
Model confidence: 100.0%

Interpretation:
This output is a decision-support signal, not a final verdict.

For the educator:
- Use this prediction as a prompt for supportive outreach.
- Review the student's context before taking action.
- Do not rely only on the model score.
- Consider whether the model may be missing relevant information.
- Document any intervention decision made after reviewing the prediction.


In [37]:
print(
    generate_educator_briefing(
        model_summary=(
            "Early-warning model trained on OULAD educational data. "
            "The model predicts whether a student is likely to be Pass or At-Risk."
        )
    )
)

EDUCATOR BRIEFING

Purpose:
The early-warning model is designed to support timely academic intervention.
It should not replace educator judgment.

Responsible use checklist:
- Has the prediction been reviewed by a human advisor or educator?
- Is the intervention supportive rather than punitive?
- Has the student been given a chance to provide context?
- Are there signs that the model may be less accurate for some groups?
- Is the decision documented in a way that can be reviewed later?

Model summary:
Early-warning model trained on OULAD educational data. The model predicts whether a student is likely to be Pass or At-Risk.


In [38]:
generate_comprehension_check(educator_role)

,question,correct_answer
0,Is the model prediction a final decision?,No. It is a decision-support signal that requi...
1,Can the model fully understand personal circum...,"No. The model may miss health, family, financi..."
2,Should an 'At-Risk' prediction be used for pun...,"No. It should be used to offer support, not pu..."
3,What should an educator do before acting on a ...,Review the student's context and combine the m...


## Final Interpretation

This notebook demonstrates two role-based views of the Awareness & Literacy layer.

The student view focuses on individual understanding. It explains that an "At-Risk" prediction is not a final judgment, but a signal that support may be useful. It also emphasizes student rights, including the right to request human review and ask what information contributed to the prediction.

The educator view includes cohort-level information such as the number of predicted at-risk students and the highest-risk cases for supportive review. These outputs are shown only in the educator view because they are intended for responsible academic support planning, not for student ranking or punishment.

The Awareness & Literacy layer does not change the model prediction. Its role is to make the prediction understandable, contestable, and safer to interpret. This supports the project goal of helping students and educators understand how AI-based early-warning systems operate, how data are used, and what limitations or biases may exist.